In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
#from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

import os
from typing import List

In [8]:
import os
os.environ["OPENAI_API_KEY"] ="Your API Key"
persist_directory = "./chroma_db"

In [9]:
class Raglcel:
  def __init__(self, chunk_size = 400, chunk_overlap = 20):
    self.chunk_size = chunk_size,
    self.chunk_overlap = chunk_overlap,
    self.text_splitters = RecursiveCharacterTextSplitter(
        chunk_overlap = chunk_overlap,
        chunk_size = chunk_size
    )

  def processor(self, path_file :str)->List[Document]:
    lod = PyPDFLoader(path_file)
    data = lod.load()

    processod_chunks = []
    for i, docs in enumerate(data):
      cleaned_data = " ".join(docs.page_content.split())
      chunks = self.text_splitters.create_documents([cleaned_data])
      processod_chunks.extend(chunks)
    return processod_chunks


In [10]:
rag = Raglcel()
chunks = rag.processor("/content/Rama_V_MLE.pdf")

In [11]:
chunks

[Document(metadata={}, page_content='RAMA KRISHNA Denton, TX| ramadev4248@gmail.com | + 1 (940)-340-9284 | LinkedIn| GitHub EDUCATION University Of North Texas, Denton, TX January 2024 – May 2025 Master in Computer and Information Sciences GPA: 4.0/4.0 Relevant Coursework: Machine Learning, Artificial Intelligence, Distributed Systems, Database Management, Operating Systems. PROFESSIONAL SUMMARY AI/ML Engineer with a passion for'),
 Document(metadata={}, page_content='with a passion for crafting scalable, practical AI solutions, leveraging over four years of experience in big data and machine learning. I am an expert in agentic AI applications using LangChain, LangGraph, N8N, CrewAI, and GPT-4, with proficiency in Vertex AI, Big Query, and AWS SageMaker for model development and deployment. I am skilled in integrating data ecosystems with Chroma DB, Faiss, and'),
 Document(metadata={}, page_content='DB, Faiss, and Pinecone for efficient vector storage and retrieval. Experienced in buil

In [12]:
print(f"chunks are: {len(chunks)} ...")
print(f"content: {chunks[0].page_content[:100]}")
print(f"content: {chunks[1].page_content[:100]}")

chunks are: 18 ...
content: RAMA KRISHNA Denton, TX| ramadev4248@gmail.com | + 1 (940)-340-9284 | LinkedIn| GitHub EDUCATION Uni
content: with a passion for crafting scalable, practical AI solutions, leveraging over four years of experien


In [13]:
vector_store = Chroma.from_documents(
        documents = chunks,
        embedding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2"),
        persist_directory=persist_directory)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
cnt = vector_store._collection.count()
cnt

18

In [15]:
data = vector_store.get(include=["embeddings", "documents", "metadatas"])
print(len(data["embeddings"]))
print(data["documents"][0])

18
RAMA KRISHNA Denton, TX| ramadev4248@gmail.com | + 1 (940)-340-9284 | LinkedIn| GitHub EDUCATION University Of North Texas, Denton, TX January 2024 – May 2025 Master in Computer and Information Sciences GPA: 4.0/4.0 Relevant Coursework: Machine Learning, Artificial Intelligence, Distributed Systems, Database Management, Operating Systems. PROFESSIONAL SUMMARY AI/ML Engineer with a passion for


In [16]:
retriever = vector_store.as_retriever(
        search_kwarg = {"k":3}
    )

In [17]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ebb4f3b3260>, search_kwargs={})

In [18]:
#Adding more documents to the vector DB

In [19]:
text = """
Agentic AI refers to autonomous AI systems that can plan, reason, and take action to complete complex goals with minimal human oversight,
 unlike traditional reactive AI that requires step-by-step prompts. These systems act proactively, using tools, data, and the internet
 to adapt to changing conditions and can even collaborate with other agents to achieve a common objective. Their ability to
 work independently allows for tasks to be automated more efficiently and with greater adaptability across various industries,
 such as IT support, customer service, and supply chain management.
"""

In [20]:
new_doc = Document(
    page_content= text
)

In [21]:
new_doc

Document(metadata={}, page_content='\nAgentic AI refers to autonomous AI systems that can plan, reason, and take action to complete complex goals with minimal human oversight,\n unlike traditional reactive AI that requires step-by-step prompts. These systems act proactively, using tools, data, and the internet \n to adapt to changing conditions and can even collaborate with other agents to achieve a common objective. Their ability to \n work independently allows for tasks to be automated more efficiently and with greater adaptability across various industries, \n such as IT support, customer service, and supply chain management.\n')

In [22]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 20)

In [23]:
texts = text_splitter.split_documents([new_doc])

In [24]:
texts

[Document(metadata={}, page_content='Agentic AI refers to autonomous AI systems that can plan, reason, and take action to complete complex goals with minimal human oversight,\n unlike traditional reactive AI that requires step-by-step prompts. These systems act proactively, using tools, data, and the internet \n to adapt to changing conditions and can even collaborate with other agents to achieve a common objective. Their ability to'),
 Document(metadata={}, page_content='work independently allows for tasks to be automated more efficiently and with greater adaptability across various industries, \n such as IT support, customer service, and supply chain management.')]

In [25]:
vector_store.add_documents(texts)
vector_store

In [26]:
cnts = vector_store._collection.count()
print(cnts)

20


In [27]:
#Adding conversational memeory to the RAG

In [38]:
llm = ChatOpenAI(model = "gpt-4o-mini",temperature = 0)

In [39]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage

In [42]:
contextualize_system_prompt = """
Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])


In [43]:
history_aware_retriever  = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ebb4f3b3260>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AI

In [44]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

system_prompt =  """
You are an assistant for question and answer task.
Use the following retrieved context to answer the question
If you don't know the answer say don't know
keep answer in 3 sentences and precise

context:{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])

answer_chain = create_stuff_documents_chain(llm, qa_prompt)

conversationl_chain = create_retrieval_chain(history_aware_retriever, answer_chain)



In [46]:
chat_history = []

result = conversationl_chain.invoke({
    "chat_history" : chat_history,
    "input" : "what is cI/CD"
})

result["answer"]

'CI/CD stands for Continuous Integration and Continuous Deployment. It is a set of practices that enable development teams to deliver code changes more frequently and reliably by automating the integration and deployment processes. CI focuses on automatically testing and merging code changes, while CD ensures that these changes are automatically deployed to production environments.'

In [47]:
chat_history.extend([
    HumanMessage(content  = "what is cI/CD" ),
    AIMessage(content = result["answer"])
])

chat_history

[HumanMessage(content='what is cI/CD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='CI/CD stands for Continuous Integration and Continuous Deployment. It is a set of practices that enable development teams to deliver code changes more frequently and reliably by automating the integration and deployment processes. CI focuses on automatically testing and merging code changes, while CD ensures that these changes are automatically deployed to production environments.', additional_kwargs={}, response_metadata={})]

In [48]:
result1 = conversationl_chain.invoke({
    "chat_history" : chat_history,
    "input" : "how was it deploys"
})

result1["answer"]

'CI/CD is deployed through automated pipelines that manage the process of integrating code changes and deploying them to production. Developers commit their code to a version control system, triggering automated tests and builds in the CI pipeline. Once the code passes all tests, it is automatically deployed to production in the CD pipeline, ensuring a seamless and efficient release process.'

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [10]:
custom_prompt =ChatPromptTemplate.from_template( """
You are an assistant for question and answer task.
Use the following retrieved context to answer the question
If you don't know the answer say don't know
keep answer in 3 sentences and precise

context:{context}

Question : {question}

Answer :""")

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question and answer task.\nUse the following retrieved context to answer the question\nIf you don't know the answer say don't know\nkeep answer in 3 sentences and precise\n\ncontext:{context}\n\nQuestion : {question}\n\nAnswer :"), additional_kwargs={})])

In [11]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [13]:
rag_chain_lcel = (
    {
        "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
    }
    |custom_prompt
    |llm
    |StrOutputParser()
)

In [14]:
response = rag_chain_lcel.invoke("What you know about CI/CD")
response


'CI/CD stands for Continuous Integration and Continuous Deployment, which are practices aimed at automating the software development process. CI involves regularly merging code changes into a central repository, where automated builds and tests are run to ensure code quality. CD extends this by automatically deploying code changes to production after passing tests, facilitating faster and more reliable software releases.'

In [18]:
def askquestion(question):
  answer = rag_chain_lcel.invoke(question)
  print(answer)

In [19]:
askquestion("what do you know about NLP")

Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves techniques for understanding, interpreting, and generating human language, utilizing methods such as stemming, lemmatization, and TF-IDF for tasks like text classification and similarity. NLP applications can range from chatbots to sentiment analysis and language translation, leveraging machine learning frameworks like TensorFlow and PyTorch.
